# 1_edger_vscontrol.R

In [1]:
#######basic setting
wd="/Users/jiahuiji/Library/CloudStorage/Dropbox/lmu_project/xeno/data/pesudo_bulk/cell_states"
setwd(wd)

#load required packages
library(edgeR)
library(stringr)
library(limma)
library(ggplot2)

output_dic="/Users/jiahuiji/Library/CloudStorage/Dropbox/lmu_project/xeno/results/de_edger/"

Loading required package: limma



In [2]:
celltype_list=list.files()

In [9]:
compare_group="Transplanted-Control"
compare_column_name="Transplant"
control_group_name="Control"
genotype_group_name="Transplanted"

## No regress co varients

In [ ]:
for (j in 1:length(celltype_list))
{
celltype=celltype_list[j]
print(celltype)

#######load data
pesudo_file=celltype
pesudo_count=read.table(file=pesudo_file, header=T, sep=",")
pesudo=pesudo_count[, -1]
rownames(pesudo)=pesudo_count$Sample_ID
#rownames(pesudo_count)=pesudo_count$X
#pesudo_count=pesudo_count[, -1]

#add meta information
clinical_info_df=read.table(file="/Users/jiahuiji/Library/CloudStorage/Dropbox/lmu_project/xeno/data/pesudo_bulk/meta_info.csv", sep=",", header=T)
dif_meta=clinical_info_df
rownames(dif_meta)=dif_meta$Sample_ID
colnames(dif_meta)[which(colnames(dif_meta)==compare_column_name)]="Disease"
pesudo=t(pesudo)





#######select sample
sample=colnames(pesudo)[which(colSums(pesudo)>0)]
pesudo=pesudo[,sample]
dif_meta=dif_meta[sample,]

dim(pesudo)
dim(dif_meta)




#######filter genes
mat_select=pesudo

row=c()
for (i in 1:length(rownames(mat_select)))
{
    if (sum(mat_select[i,])>0)
    {row=append(row,i)}
}
mat=mat_select[row,]
meta=dif_meta
dim(mat)
dim(dif_meta)





if (length(which(meta$Disease==control_group_name))>2 & length(which(meta$Disease==genotype_group_name))>2)
{
    #######Differential expression analysis - edgeR
    #create edgeR object
    deglist=DGEList(counts=mat, group=as.factor(meta$Disease))
    colnames(deglist)=rownames(meta)

    #filter low expression gene 
    control_idx=deglist$samples$group == control_group_name
    genotype_idx=deglist$samples$group == genotype_group_name
    # Calculate mean counts per gene per group
    mean_control=rowMeans(deglist$counts[, control_idx])
    mean_genotype=rowMeans(deglist$counts[, genotype_idx])
    # Apply filter
    keep=which(mean_control > 0.0125 | mean_genotype > 0.0125)


    deglist_keep=deglist[keep, keep.lib.sizes = FALSE]
    deglist_norm=calcNormFactors(deglist_keep, method="TMM")

    design=model.matrix(~0 + as.factor(meta$Disease))
    print(head(design))
    meta_name=colnames(design)
    cleaned_col_names=gsub(".*\\)", "", meta_name)
    colnames(design)[1:2]=cleaned_col_names[1:2]

    deg=estimateDisp(deglist_norm, design, robust=TRUE)
    fit=glmQLFit(deg, design)
    compare=makeContrasts(compare_group, levels=design)
    qlf=glmQLFTest(fit, contrast=compare)
    topTags(qlf)
    res=topTags(qlf, n = nrow(deglist$counts))$table
    dim(res)

    #add information of number of samples in each comparison group
    res[,paste0("Observations_", genotype_group_name)]=as.numeric(table(meta[,"Disease"])[genotype_group_name])
    res[,paste0("Observations_", control_group_name)]=as.numeric(table(meta[,"Disease"])[control_group_name])

    write.table(res, file=paste0(output_dic, "edger_de_", celltype), sep="\t", quote=F)
}
}

[1] "pseudobulk_Angiogenic_EC-1_arterial_.csv"
  as.factor(meta$Disease)Control as.factor(meta$Disease)Transplanted
1                              1                                   0
2                              1                                   0
3                              1                                   0
4                              1                                   0
5                              0                                   1
6                              0                                   1
[1] "pseudobulk_Angiogenic_EC-2_hypoxic_.csv"
  as.factor(meta$Disease)Control as.factor(meta$Disease)Transplanted
1                              1                                   0
2                              1                                   0
3                              1                                   0
4                              1                                   0
5                              0                                   1
6         

## Regress covarient

In [ ]:
#celltype_list=c('pseudobulk_unclassified_sus.csv', 'pseudobulk_vCM1.csv', 'pseudobulk_vCM2_ISG.csv', 'pseudobulk_vCM3.csv', 'pseudobulk_vCM4.csv', 'pseudobulk_vCM5.csv', 'pseudobulk_vFB1.csv', 'pseudobulk_vFB10.csv', 'pseudobulk_vFB2.csv', 'pseudobulk_vFB3.csv', 'pseudobulk_vFB4.csv', 'pseudobulk_vFB5.csv', 'pseudobulk_vFB6.csv', 'pseudobulk_vFB7.csv', 'pseudobulk_vFB8.csv', 'pseudobulk_vFB9.csv')

for (j in 1:length(celltype_list))
{
celltype=celltype_list[j]
print(celltype)

#######load data
pesudo_file=celltype
pesudo_count=read.table(file=pesudo_file, header=T, sep=",")
pesudo=pesudo_count[, -1]
rownames(pesudo)=pesudo_count$Sample_ID
#rownames(pesudo_count)=pesudo_count$X
#pesudo_count=pesudo_count[, -1]

#add meta information
clinical_info_df=read.table(file="/Users/jiahuiji/Library/CloudStorage/Dropbox/lmu_project/xeno/data/pesudo_bulk/meta_info.csv", sep=",", header=T)
dif_meta=clinical_info_df
rownames(dif_meta)=dif_meta$Sample_ID
colnames(dif_meta)[which(colnames(dif_meta)==compare_column_name)]="Disease"
pesudo=t(pesudo)





#######select sample
sample=colnames(pesudo)[which(colSums(pesudo)>0)]
pesudo=pesudo[,sample]
dif_meta=dif_meta[sample,]

dim(pesudo)
dim(dif_meta)




#######filter genes
mat_select=pesudo

row=c()
for (i in 1:length(rownames(mat_select)))
{
    if (sum(mat_select[i,])>0)
    {row=append(row,i)}
}
mat=mat_select[row,]
meta=dif_meta
dim(mat)
dim(dif_meta)





if (length(which(meta$Disease==control_group_name))>2 & length(which(meta$Disease==genotype_group_name))>2)
{
    #######Differential expression analysis - edgeR
    #create edgeR object
    deglist=DGEList(counts=mat, group=as.factor(meta$Disease))
    colnames(deglist)=rownames(meta)


    #filter low expression gene 
    control_idx=deglist$samples$group == control_group_name
    genotype_idx=deglist$samples$group == genotype_group_name
    # Calculate mean counts per gene per group
    mean_control=rowMeans(deglist$counts[, control_idx])
    mean_genotype=rowMeans(deglist$counts[, genotype_idx])
    # Apply filter
    keep=which(mean_control > 0.0125 | mean_genotype > 0.0125)


    deglist_keep=deglist[keep, keep.lib.sizes = FALSE]
    deglist_norm=calcNormFactors(deglist_keep, method="TMM")

    design=model.matrix(~0 + as.factor(meta$Disease) + as.numeric(meta$Survival))
    print(head(design))
    meta_name=colnames(design)
    cleaned_col_names=gsub(".*\\)", "", meta_name)
    colnames(design)[1:2]=cleaned_col_names[1:2]
    colnames(design)[3]="Survival"

    deg=estimateDisp(deglist_norm, design, robust=TRUE)
    fit=glmQLFit(deg, design)
    compare=makeContrasts(compare_group, levels=design)
    qlf=glmQLFTest(fit, contrast=compare)
    topTags(qlf)
    res=topTags(qlf, n = nrow(deglist$counts))$table
    dim(res)

    #add information of number of samples in each comparison group
    res[,paste0("Observations_", genotype_group_name)]=as.numeric(table(meta[,"Disease"])[genotype_group_name])
    res[,paste0("Observations_", control_group_name)]=as.numeric(table(meta[,"Disease"])[control_group_name])

    write.table(res, file=paste0(output_dic, "edger_de_", celltype), sep="\t", quote=F)
}
}

[1] "pseudobulk_Angiogenic_EC-1_arterial_.csv"
  as.factor(meta$Disease)Control as.factor(meta$Disease)Transplanted
1                              1                                   0
2                              1                                   0
3                              1                                   0
4                              1                                   0
5                              0                                   1
6                              0                                   1
  as.numeric(meta$Survival)
1                         0
2                         0
3                         0
4                         0
5                        90
6                        51
[1] "pseudobulk_Angiogenic_EC-2_hypoxic_.csv"
  as.factor(meta$Disease)Control as.factor(meta$Disease)Transplanted
1                              1                                   0
2                              1                                   0
3                    